In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
churn=pd.read_csv("Clean_TelcoCustomerChurn.csv")

In [3]:
churn.sample(7)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
1564,Male,1,No,No,10,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.35,Yes
6315,Female,0,Yes,Yes,48,Yes,No,DSL,Yes,Yes,Yes,No,No,Yes,Two year,Yes,Credit card (automatic),70.10,No
5259,Male,1,Yes,No,30,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,No,Mailed check,77.90,No
350,Male,0,Yes,Yes,37,Yes,Yes,DSL,Yes,No,Yes,Yes,No,No,Two year,Yes,Credit card (automatic),62.80,No
6791,Male,0,No,No,19,No,No,DSL,No,No,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,39.65,Yes
583,Female,0,Yes,Yes,1,Yes,No,No,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,19.00,No
3888,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,Yes,Month-to-month,No,Credit card (automatic),80.15,Yes


In [4]:
churn.shape

(7043, 19)

In [5]:
churn["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [6]:
X=churn.drop(columns="Churn")
y=churn.Churn

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)

# Pipeline

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,FunctionTransformer,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [9]:
churn.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'Churn'],
      dtype='str')

In [10]:
num_col=["tenure","MonthlyCharges"]
cat_col=['gender', 'SeniorCitizen', 'Partner', 'Dependents','PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV','StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

In [11]:
num_pipe=Pipeline(steps=[
    ("Encode",StandardScaler())
])
cat_pipe=Pipeline(steps=[
    ("Encode",OneHotEncoder(drop="first"))
])

In [12]:
preprocessing=ColumnTransformer(transformers=[
    ("prepro_num",num_pipe,num_col),
    ("prepro_cat",cat_pipe,cat_col)
])

In [13]:
full_pipe=Pipeline(steps=[
    ("preprocessing",preprocessing),
    ("Model",LogisticRegression())
    # ("Model1",DecisionTreeClassifier(max_depth=11))
])

In [14]:
full_pipe.fit(X_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('prepro_num',
                                                  Pipeline(steps=[('Encode',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges']),
                                                 ('prepro_cat',
                                                  Pipeline(steps=[('Encode',
                                                                   OneHotEncoder(drop='first'))]),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('Model', LogisticRegression())])

In [15]:
a=full_pipe.named_steps["Model"]

In [16]:
a.coef_[0]   # LogisticRegression has coef_, not feature_importances_ (that's a tree-model attribute)

array([-0.77821709, -0.04658223,  0.027259  ,  0.14497175, -0.00597722,
       -0.27351951, -0.3122409 ,  0.33903302,  0.86015796, -0.9743184 ,
       -0.3773226 , -0.12665119, -0.00737449, -0.32722859,  0.24738208,
        0.3367479 , -0.71797619, -1.27827375,  0.40532382, -0.04441324,
        0.38002371,  0.12523773])

In [17]:
asdf=full_pipe.named_steps["preprocessing"]

In [18]:
asdf.get_feature_names_out()

array(['prepro_num__tenure', 'prepro_num__MonthlyCharges',
       'prepro_cat__gender_Male', 'prepro_cat__SeniorCitizen_1',
       'prepro_cat__Partner_Yes', 'prepro_cat__Dependents_Yes',
       'prepro_cat__PhoneService_Yes', 'prepro_cat__MultipleLines_Yes',
       'prepro_cat__InternetService_Fiber optic',
       'prepro_cat__InternetService_No', 'prepro_cat__OnlineSecurity_Yes',
       'prepro_cat__OnlineBackup_Yes', 'prepro_cat__DeviceProtection_Yes',
       'prepro_cat__TechSupport_Yes', 'prepro_cat__StreamingTV_Yes',
       'prepro_cat__StreamingMovies_Yes', 'prepro_cat__Contract_One year',
       'prepro_cat__Contract_Two year',
       'prepro_cat__PaperlessBilling_Yes',
       'prepro_cat__PaymentMethod_Credit card (automatic)',
       'prepro_cat__PaymentMethod_Electronic check',
       'prepro_cat__PaymentMethod_Mailed check'], dtype=object)

In [19]:
# from sklearn.tree import plot_tree

# plot_tree(a,feature_names=full_pipe.named_steps['preprocessing'].get_feature_names_out(),filled=True, 
#     rounded=True, );

In [20]:
y_pred=full_pipe.predict(X_test)

In [21]:
y_test

4994    Yes
6828     No
755     Yes
404      No
981      No
       ... 
4373     No
303      No
291      No
6727     No
337      No
Name: Churn, Length: 2113, dtype: str

In [22]:
# y_pred.tolist()

In [23]:
full_pipe.score(X_train,y_train)

0.8030425963488844

In [24]:
full_pipe.score(X_test,y_test)

0.7998106956933271

In [25]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix,roc_auc_score,recall_score,precision_score

In [26]:
(cross_val_score(full_pipe,X_train,y_train,cv=5)).mean()

np.float64(0.8022312373225153)

In [27]:

y_pred=full_pipe.predict(X_test)

In [28]:
confusion_matrix(y_test,y_pred)

array([[1387,  165],
       [ 258,  303]])

In [29]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

          No       0.84      0.89      0.87      1552
         Yes       0.65      0.54      0.59       561

    accuracy                           0.80      2113
   macro avg       0.75      0.72      0.73      2113
weighted avg       0.79      0.80      0.79      2113



In [30]:
import pickle as pkl

In [31]:
pkl.dump({
    "Model":full_pipe,
    "cat_col":cat_col,
    "num_col":num_col
},open("churn_model.pkl","wb"))

In [32]:
roc_auc_score(y_test, full_pipe.predict_proba(X_test)[:,1])   # needs P(Yes), not the predicted labels

np.float64(0.8432176525718066)

In [33]:
import pandas as pd

# 1. Get the feature names output by your preprocessing pipeline step
feature_names = full_pipe.named_steps["preprocessing"].get_feature_names_out()

# 2. Match the names with the importance array shown in your image
importance_df = pd.DataFrame({
    'Feature Name': feature_names,
    'Importance': a.coef_[0]
})

# 3. Sort them so the most important features appear at the top
importance_df = importance_df.sort_values(by='Importance', key=abs, ascending=False).reset_index(drop=True)  # sort by size of effect, not just sign

# 4. Display the results
importance_df


,Feature Name,Importance
0,prepro_cat__Contract_Two year,-1.278274
1,prepro_cat__InternetService_No,-0.974318
2,prepro_cat__InternetService_Fiber optic,0.860158
3,prepro_num__tenure,-0.778217
4,prepro_cat__Contract_One year,-0.717976
5,prepro_cat__PaperlessBilling_Yes,0.405324
6,prepro_cat__PaymentMethod_Electronic check,0.380024
7,prepro_cat__OnlineSecurity_Yes,-0.377323
8,prepro_cat__MultipleLines_Yes,0.339033
9,prepro_cat__StreamingMovies_Yes,0.336748
